# MSU Chat Board — RAG Pipeline Reference Notebook

This notebook documents, step by step, how the **Midlands State University (MSU) Chat Board** is built.
It is a **Retrieval-Augmented Generation (RAG)** system: instead of relying only on what a language model
already knows, we first *retrieve* relevant passages from official MSU documents and then ask the model to
*answer using those passages*.

**Pipeline overview**

| Stage | What happens | Code |
|---|---|---|
| 1 | Load & clean the MSU PDFs | `src/data_loader.py` |
| 2 | Split pages into chunks | `src/embedding.py` |
| 3 | Turn chunks into embeddings (vectors) | `src/embedding.py` |
| 4 | FAISS vector index + BM25 keyword index, routed retrieval | `src/vectorstore.py` |
| 5 | Unseen retrieval evaluation (frozen questions, blind grading) | — |
| 6 | Evidence-based retrieval / ranking improvements | `src/vectorstore.py` |
| 7 | LLM answer generation: connect the LLM & write the prompt | `src/search.py` |
| 8 | Answers with source citations | `src/search.py` |
| 9 | Chat interface | `app.py` |
| 10 | End-to-end testing | — |

> This notebook calls the **same functions** the app uses (from the `src/` package), so what you see here is
> exactly what the chat board does. Sections are added as each stage is completed and approved.

## 0. Setup

Run this notebook from the project folder (`RAG-Tutorials-main`) so that `import src...` works.

In [1]:
# --- Standard library imports ---
import os            # file-system paths
import sys           # to make sure the project folder is importable
import time          # to time how long loading takes
import collections   # Counter, for quick statistics

# Make the project root importable, so `from src.xxx import ...` works
# even if Jupyter was started from a different folder.
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

# Auto-reload: when a file in src/ is edited, Jupyter re-imports it before running each cell.
# Without this, the kernel keeps using the OLD version of src/ modules until it is restarted
# (which causes errors like "cannot import name 'EMBEDDING_MODEL'").
try:
    ipython = get_ipython()                       # only exists inside Jupyter
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")     # 2 = reload all modules every time
    print("Autoreload enabled")
except NameError:
    pass                                          # running as plain Python, nothing to do

Project root: C:\Users\donal\OneDrive\Desktop\RAG\RAG-Tutorials-main
Autoreload enabled


## Stage 1 — Loading & cleaning the MSU documents

**Goal:** read every page of the MSU yearbooks into memory as clean text, and remember *where* each page came
from (book + page number) so the chat board can cite its sources later.

**Data source:** `C:\Users\donal\OneDrive\Desktop\RAG\msu docs`
- *HUMANITIES – General Information and Regulations* (931 pages)
- *SCIENCES – General Information and Regulations Vol. 2* (1,132 pages)

**Key decisions**
1. **PyMuPDF instead of pypdf** — reads each book in ~4 s instead of ~2 min, and copes better with these PDFs.
2. **Text cleaning** — the PDFs contain a strange "one-dot leader" character (`․`) instead of full stops,
   invisible "word joiner" characters, curly quotes, dot leaders in the table of contents, and a printed page
   number at the top of each page. All of these are normalised.
3. **Metadata** — every page is stored as a LangChain `Document` with `source`, `file_name` and `page`.
4. **De-duplication** — both books repeat the same front matter (the MSU Act, statutes, general regulations).
   Pages that are *exactly* identical are kept only once, so the chat board does not retrieve the same passage twice.
5. **Blank pages skipped** — pages with fewer than 30 characters (covers, separators) are ignored.

### 1.1 Look at the raw PDFs

Before cleaning anything, open the PDFs directly to see how many pages they have and what the raw text looks like.

In [2]:
from pathlib import Path
import pymupdf  # PyMuPDF — fast PDF text extraction

# Import the folder setting from our loader so the notebook and app always use the same path
from src.data_loader import MSU_DOCS_DIR

# List every PDF in the MSU docs folder
pdf_files = sorted(Path(MSU_DOCS_DIR).glob("*.pdf"))
for pdf_file in pdf_files:
    with pymupdf.open(pdf_file) as pdf:            # open the PDF (closed automatically afterwards)
        print(f"{pdf_file.name}  ->  {pdf.page_count} pages")

C:\Users\donal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HUMANITIES -GENERAL INFORMATION AND REGULATIONS final done (1).pdf  ->  931 pages
SCIENCES - GENERAL INFORMATION AND REGULATIONS VOLUME2 final done (1).pdf  ->  1132 pages


In [3]:
# Show the RAW text of one page exactly as PyMuPDF extracts it.
# repr() makes hidden characters visible: '\n' line breaks and the odd '\u2024' full stops.
sample_pdf = pdf_files[0]          # Humanities yearbook
with pymupdf.open(sample_pdf) as pdf:
    raw_text = pdf[10].get_text()  # page index 10 = PDF page 11 (Python counts from 0)

print(repr(raw_text[:400]))

'6 \n \n27․ \nStudent Disciplinary Committee․ \n28․ \nFinance Committee․ \n28A․ \nService of process․ \n29․ \nStatutes․ \n30․ \nRegulations․ \n31․ \nValidity of decisions of Council, Senate, Convocation and boards and committees․ \nSCHEDULE: Statutes of the University․ \nACT \nTo establish the Midlands State University and to provide for matters connected therewith or incidental thereto․ \n[amended by General Laws '


In [4]:
# Count the unusual (non-ASCII) characters across the whole book,
# so we know which ones the cleaning step must handle.
with pymupdf.open(sample_pdf) as pdf:
    all_text = "".join(page.get_text() for page in pdf)

odd_chars = collections.Counter(ch for ch in all_text if ord(ch) > 127)
for ch, count in odd_chars.most_common(10):
    print(f"U+{ord(ch):04X}  {ch!r:8}  x{count}")

U+2022  '•'       x1680
U+2019  '’'       x1389
U+2024  '․'       x364
U+2026  '…'       x363
U+2018  '‘'       x252
U+201D  '”'       x154
U+201C  '“'       x152
U+2013  '–'       x117
U+25CF  '●'       x43
U+2014  '—'       x41


### 1.2 The cleaning function

`clean_page_text()` (in `src/data_loader.py`) does, in order:
1. **Character fixes** — `․`→`.`, removes invisible characters, curly quotes → straight quotes, dashes → `-`.
2. **Removes the printed page number** at the start of the page (e.g. `"146 \n"`).
3. **Collapses table-of-contents dot leaders** (`.........` or `………`) into `" ... "`.
4. **Joins lone list markers to their text** — the PDFs put bullets and numbers on their own line
   (`"6.\nLEARNING OUTCOMES"`, `"•\nIdentify pertinent issues"`). There were 5,323 such lines; joining them
   (`"6. LEARNING OUTCOMES"`) keeps headings and bullets intact when the text is chunked in Stage 2.
5. **Tidies whitespace** — trailing/leading spaces, repeated spaces, and multiple blank lines.

Compare the same page before and after:

In [5]:
from src.data_loader import clean_page_text

cleaned_text = clean_page_text(raw_text)   # apply all cleaning rules to the raw page

print("===== BEFORE =====")
print(raw_text[:300])
print("===== AFTER =====")
print(cleaned_text[:300])

===== BEFORE =====
6 
 
27․ 
Student Disciplinary Committee․ 
28․ 
Finance Committee․ 
28A․ 
Service of process․ 
29․ 
Statutes․ 
30․ 
Regulations․ 
31․ 
Validity of decisions of Council, Senate, Convocation and boards and committees․ 
SCHEDULE: Statutes of the University․ 
ACT 
To establish the Midlands State Univers
===== AFTER =====
27. Student Disciplinary Committee.
28. Finance Committee.
28A. Service of process.
29. Statutes.
30. Regulations.
31. Validity of decisions of Council, Senate, Convocation and boards and committees.
SCHEDULE: Statutes of the University.
ACT
To establish the Midlands State University and to provide 


### 1.3 Load everything

`load_all_documents()` loops over every PDF (plus any TXT / CSV / Word files in the folder), cleans each page,
attaches metadata, skips blank pages, and removes exact duplicate pages.

In [6]:
from src.data_loader import load_all_documents

start = time.time()
docs = load_all_documents()        # defaults to MSU_DOCS_DIR, with de-duplication switched on
print(f"Loading took {time.time() - start:.1f} seconds")

[INFO] Data path: C:\Users\donal\OneDrive\Desktop\RAG\msu docs
[INFO] Found 2 PDF files


[INFO] Loaded 927 pages from HUMANITIES -GENERAL INFORMATION AND REGULATIONS final done (1).pdf


[INFO] Loaded 1128 pages from SCIENCES - GENERAL INFORMATION AND REGULATIONS VOLUME2 final done (1).pdf


[INFO] Removed 99 duplicate pages
[INFO] Total loaded documents: 1956
Loading took 15.8 seconds


### 1.4 Inspect the result

In [7]:
# How many pages came from each book (after removing duplicates)?
pages_per_source = collections.Counter(doc.metadata["source"] for doc in docs)
for source, count in pages_per_source.items():
    print(f"{source:35} {count} pages")

# Total amount of text we will be searching over
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"\nTotal: {len(docs)} pages, {total_chars:,} characters")

MSU Humanities Yearbook             927 pages
MSU Sciences Yearbook Vol. 2        1029 pages

Total: 1956 pages, 5,797,457 characters


In [8]:
# Each item is a LangChain Document: the text (page_content) + metadata (where it came from).
example = docs[400]
print("Metadata:", example.metadata)   # used later for citations, e.g. "MSU Humanities Yearbook, p. 404"
print("\nText:\n", example.page_content[:500])

Metadata: {'source': 'MSU Humanities Yearbook', 'file_name': 'HUMANITIES -GENERAL INFORMATION AND REGULATIONS final done (1).pdf', 'page': 404}

Text:
 BSS204 World Political Systems
The module focuses on the evolvement of world political systems; Greek and Roman origins of concepts of democracy;
Western political thinkers and the evolution of democratic institutions; liberal democracy and constitutionalism;
communism and socialism; militarism and dictatorship; global unity; regional groupings and integration.
BSS205 Poverty and development

The module seeks to develop an integrated approach to the theory and practice in developmental issues, w


In [9]:
# Sanity check: none of the problem characters should survive cleaning.
problem_chars = "\u2024\u2060\u2018\u2019\u201c\u201d"
leftover = collections.Counter(ch for doc in docs for ch in doc.page_content if ch in problem_chars)
print("Leftover problem characters:", dict(leftover) or "none ✅")

Leftover problem characters: none ✅


In [10]:
# Check the list-marker fix: count lines that still contain ONLY a bullet or list number.
# (Before the fix there were 5,323; the few left are followed by a blank line, so they are left alone.)
import re
lone_marker = re.compile(r"^(•|o|-|\d{1,3}[A-Z]?\.(?:\d{1,3}\.?)*|[a-zA-Z][.)]|[ivxlc]{1,5}[.)])$")
remaining = sum(1 for doc in docs for line in doc.page_content.split("\n") if lone_marker.match(line.strip()))
print("Lone marker lines remaining:", remaining)

Lone marker lines remaining: 8


**Stage 1 result:** ~1,956 clean pages (927 Humanities + 1,029 Sciences after removing 99 duplicate pages),
each tagged with its source book and page number, loaded in ~10–20 seconds.

## Stage 2 — Chunking

**Goal:** split each page into smaller pieces ("chunks"). Later, the chat board searches over chunks, not whole
pages, so each chunk should hold **one focused idea** — one course description or one regulation clause.

**Why not use whole pages?** A typical page is ~3,000 characters and mixes several courses or rules. Searching
whole pages returns lots of irrelevant text, and embedding models (Stage 3) only read roughly the first
~1,000 characters anyway.

**Key decisions**
1. **Chunk size 800 characters, overlap 150.** Measured on the MSU text: 90% of paragraphs are ≤ ~620
   characters, so most course descriptions fit whole in one chunk. The overlap means text cut at a boundary
   also appears at the start of the next chunk, so nothing is lost.
2. **Smart split points** — `RecursiveCharacterTextSplitter` tries to cut at a blank line first, then a line
   break (each course starts on a new line), then a sentence end, then a space.
3. **Citation header** — each chunk starts with e.g. `[MSU Humanities Yearbook, p. 404]` so the LLM knows the
   source of every passage it reads.
4. **Metadata kept** — `source`, `page` (and a new `chunk_id`) are copied to every chunk.

### 2.1 Why 800 characters? Measure the paragraphs

In [11]:
import statistics

# Split every page into paragraphs (separated by blank lines) and measure their lengths
paragraph_lengths = sorted(
    len(p) for doc in docs for p in doc.page_content.split("\n\n") if p.strip()
)

print("Paragraphs:", len(paragraph_lengths))
print("Median length:", statistics.median(paragraph_lengths))
print("90% of paragraphs are shorter than:", paragraph_lengths[int(0.9 * len(paragraph_lengths))])

Paragraphs: 24492
Median length: 60.0
90% of paragraphs are shorter than: 616


### 2.2 Split the pages into chunks

`chunk_documents()` lives in `src/embedding.py`. `CHUNK_SIZE` and `CHUNK_OVERLAP` are defined at the top of that
file, so they can be tuned in one place.

In [12]:
from src.embedding import chunk_documents, CHUNK_SIZE, CHUNK_OVERLAP

print(f"Chunk size: {CHUNK_SIZE} characters, overlap: {CHUNK_OVERLAP} characters")

chunks = chunk_documents(docs)    # pages -> chunks (metadata copied, header added)

Chunk size: 800 characters, overlap: 150 characters


[INFO] Split 1956 documents into 10732 chunks.


### 2.3 Inspect the chunks

In [13]:
chunk_lengths = sorted(len(chunk.page_content) for chunk in chunks)

print("Total chunks:", len(chunks))
print("Average length:", round(statistics.mean(chunk_lengths)))
print("Smallest / largest:", chunk_lengths[0], "/", chunk_lengths[-1])

# Chunks per book
print(dict(collections.Counter(chunk.metadata["source"] for chunk in chunks)))

# Very small chunks are usually lone headings (e.g. 'CONTENTS') — only ~1% of the total
print("Tiny chunks (< 120 chars):", sum(1 for n in chunk_lengths if n < 120))

Total chunks: 10732
Average length: 610
Smallest / largest: 40 / 839
{'MSU Humanities Yearbook': 5614, 'MSU Sciences Yearbook Vol. 2': 5118}
Tiny chunks (< 120 chars): 131


In [14]:
# Look at one chunk in full: the citation header is the first line,
# and the metadata travels with it for later use.
example_chunk = next(c for c in chunks if "MPG 704" in c.page_content)
print("Metadata:", {k: v for k, v in example_chunk.metadata.items() if k != "file_name"})
print()
print(example_chunk.page_content)

Metadata: {'source': 'MSU Humanities Yearbook', 'page': 885, 'chunk_id': 5301}

[MSU Humanities Yearbook, p. 885]
Written Examinations: End-of-semester examinations constitute 60% of the final mark.

9.0 PROVISION FOR PROGRESSION
9.1 Refer to Section 6 of the Faculty Regulations

10.0 FAILURE SATISFY EXAMINERS
Refer to Section 9 of the General Regulations for Undergraduate Degrees.

11.0. GRADING AND DEGREE CLASSIFICATION
Refer to Section 10 of the Faculty Regulations

12.0 DEGREE WEIGHTING
Refer to Section 11 of the Faculty Regulations.

13.0 PROGRAMME STRUCTURE

PROGRAMME STRUCTURE
Level 1 Semester 1
MPG 701 Theories and Concepts of Gender

18
MPG 702 Gender and Security Sector

18
MPG 703 Gender, Peace and Security

18
MPG 704 Research Methods in Gender, Peace and Security

18
MPG 705 Religion, Gender and Ethics in Africa

18

Electives
MPG 706 Human Security and Displacement


In [15]:
# Show how the overlap works: the end of one chunk is repeated at the start of the next
# (only when both chunks come from the same page).
first, second = chunks[5000], chunks[5001]
print("...END OF CHUNK 5000:\n", first.page_content[-200:])
print("\nSTART OF CHUNK 5001:\n", second.page_content[:250])

...END OF CHUNK 5000:
 OR MASTER OF SCIENCE IN HUMAN CAPITAL MANAGEMENT DEGREE (MHCD)

Duration:

18 months

Actual Load:

288
Minimum Credit Load:

270
Maximum Credit Load:

360
Maximum MBKs credit Load:
216
ZNQF Level:

9

START OF CHUNK 5001:
 [MSU Humanities Yearbook, p. 835]
Duration:

18 months

Actual Load:

288
Minimum Credit Load:

270
Maximum Credit Load:

360
Maximum MBKs credit Load:
216
ZNQF Level:

9

1.0 PURPOSE OF THE PROGRAMME
To provide both learners and practising human cap


**Stage 2 result:** 10,732 chunks (average ~610 characters, max ~840), each starting with a citation header
and carrying its book and page number. Chunking takes under a second.

## Stage 3 — Embeddings

**Goal:** turn every chunk into an **embedding** — a list of numbers (a vector) that captures its *meaning*.
A user's question is turned into a vector the same way, and the chunks whose vectors are closest to the
question's vector are the most relevant ones.

**Model chosen: `BAAI/bge-small-en-v1.5`** (set once as `EMBEDDING_MODEL` in `src/embedding.py`)
- 384 numbers per vector, reads up to **512 tokens** (all our ~800-character chunks fit without being cut).
- Runs on CPU; downloaded once (~130 MB) from HuggingFace and cached.

**Key decisions**
1. **Normalised vectors** — every vector is scaled to length 1, so the dot product of two vectors equals their
   **cosine similarity** (1 = same meaning, 0 = unrelated).
2. **Query prefix** — BGE models were trained to expect questions to start with
   `"Represent this sentence for searching relevant passages: "`. `embed_query()` adds it automatically.
   Chunks (passages) are embedded *without* the prefix.

> ⏱ Embedding all 10,732 chunks takes ~32 minutes on this CPU, so this notebook demonstrates the steps on a
> sample. The full embedding is done once in Stage 4 and saved to disk.

### 3.1 Load the embedding model

In [16]:
from src.embedding import EmbeddingPipeline, EMBEDDING_MODEL

embedder = EmbeddingPipeline()   # uses EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" by default

print("Model:", embedder.model_name)
print("Vector size:", embedder.model.get_embedding_dimension())
print("Max tokens per chunk:", embedder.model.max_seq_length)
print("Query prefix:", repr(embedder.query_prefix))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2210.21it/s]

[INFO] Loaded embedding model: BAAI/bge-small-en-v1.5 (dim=384)
Model: BAAI/bge-small-en-v1.5
Vector size: 384
Max tokens per chunk: 512
Query prefix: 'Represent this sentence for searching relevant passages: '


### 3.2 What an embedding looks like

In [17]:
import numpy as np

# Embed a single chunk. The result is a 2-D array: (number of chunks, 384)
one_vector = embedder.embed_chunks([example_chunk])

print("Shape:", one_vector.shape)
print("First 8 numbers:", np.round(one_vector[0][:8], 4))
print("Vector length (should be 1.0 because it is normalised):", round(float(np.linalg.norm(one_vector[0])), 4))

[INFO] Generating embeddings for 1 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.28it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.28it/s]

[INFO] Embeddings shape: (1, 384)
Shape: (1, 384)
First 8 numbers: [-0.0013  0.0715 -0.0004 -0.0744  0.0291  0.043  -0.0047 -0.0106]
Vector length (should be 1.0 because it is normalised): 1.0


### 3.3 Similarity: does the closest chunk have the right meaning?

We embed a question and three hand-picked chunks, then compare them with cosine similarity.
The deferment chunk should score highest for a deferment question.

In [18]:
# Pick three chunks on very different topics
deferment_chunk = next(c for c in chunks if "11. Deferment of Studies" in c.page_content)
course_chunk    = next(c for c in chunks if "HAWS 433 Biotechnology" in c.page_content)
vc_chunk        = next(c for c in chunks if "Vice-Chancellor shall be appointed" in c.page_content)
candidates = [deferment_chunk, course_chunk, vc_chunk]

candidate_vectors = embedder.embed_chunks(candidates)            # shape (3, 384)
question = "How do I defer my studies?"
question_vector = embedder.embed_query(question)                 # shape (1, 384), prefix added

# Because vectors are normalised, a matrix product gives cosine similarities directly
scores = candidate_vectors @ question_vector[0]

print("Question:", question, "\n")
for chunk, score in sorted(zip(candidates, scores), key=lambda pair: -pair[1]):
    first_line = chunk.page_content.split("\n")[1][:70]           # line after the citation header
    print(f"{score:.3f}  {chunk.metadata['source']}, p.{chunk.metadata['page']}:  {first_line}")

[INFO] Generating embeddings for 3 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]

[INFO] Embeddings shape: (3, 384)
Question: How do I defer my studies? 

0.667  MSU Humanities Yearbook, p.153:  10.5.2 No break in the normal continuity of study shall be permitted, 
0.478  MSU Humanities Yearbook, p.15:  8 Vice-Chancellor
0.435  MSU Sciences Yearbook Vol. 2, p.283:  HAWS 433 Biotechnology in Animal Science


### 3.4 Embedding speed (sample) and estimate for the full collection

In [19]:
sample = chunks[:200]                    # a small sample so the notebook stays quick

start = time.time()
sample_vectors = embedder.embed_chunks(sample)
seconds = time.time() - start

per_second = len(sample) / seconds
print(f"Embedded {len(sample)} chunks in {seconds:.1f}s  ({per_second:.1f} chunks/second)")
print(f"Estimated time for all {len(chunks):,} chunks: {len(chunks) / per_second / 60:.0f} minutes")

[INFO] Generating embeddings for 200 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:  14%|█▍        | 1/7 [00:12<01:12, 12.14s/it]

Batches:  29%|██▊       | 2/7 [00:24<01:02, 12.44s/it]

Batches:  43%|████▎     | 3/7 [00:37<00:49, 12.36s/it]

...

Batches:  71%|███████▏  | 5/7 [00:53<00:19,  9.61s/it]

Batches:  86%|████████▌ | 6/7 [00:56<00:07,  7.55s/it]

Batches: 100%|██████████| 7/7 [00:57<00:00,  5.38s/it]

Batches: 100%|██████████| 7/7 [00:57<00:00,  8.26s/it]

[INFO] Embeddings shape: (200, 384)
Embedded 200 chunks in 57.8s  (3.5 chunks/second)
Estimated time for all 10,732 chunks: 52 minutes


### 3.5 Model comparison results (reference)

Before choosing the model, both candidates embedded **all 10,732 chunks** and were tested on 10 MSU questions
whose correct answers were located in the yearbooks beforehand. Each top result was checked by hand.

| | all-MiniLM-L6-v2 | **BAAI/bge-small-en-v1.5** |
|---|---|---|
| Vector size | 384 | 384 |
| Max tokens per chunk | 256 | 512 |
| Time to embed all chunks (CPU) | 892 s (~15 min) | 1,915 s (~32 min) |
| Correct top result | 5 / 10 | **6 / 10** |
| Correct answer in top 5 | 8 / 10 | **9 / 10** |

| # | Question | MiniLM top-1 | bge-small top-1 |
|---|---|---|---|
| 1 | What is HAWS 433 about? | ❌ | ❌ |
| 2 | Describe the module MPG 704 | ❌ | ❌ |
| 3 | How do I defer my studies? | ✅ | ✅ |
| 4 | How are bachelor's degrees classified? | ✅ | ✅ |
| 5 | Who appoints the Vice-Chancellor? | ❌ (Pro-VC) | ✅ |
| 6 | What does the Student Disciplinary Committee do? | ✅ | ❌ (staff committee) |
| 7 | What happens if a student is caught plagiarising? | ✅ | ✅ |
| 8 | Can I write a supplementary exam if I fail? | ✅ | ✅ |
| 9 | What is voluntary termination of studies? | ❌ | ✅ |
| 10 | Which module covers statistics for forensic science? | ❌ | ❌ |

**Findings**
- bge-small is somewhat better, so it was chosen. The extra embedding time is a one-off cost: the vectors are
  saved, and answering a question only needs to embed the question (milliseconds).
- **Both models fail on course codes** (HAWS 433, MPG 704): to an embedding model "MPG 704" carries little
  meaning. This is fixed in Stage 4 by adding **keyword search (BM25)** alongside vector search — *hybrid search*.
- Similarity scores are **not comparable across models** (bge scores run higher in general).

## Stage 4 — Vector index, keyword index & routed retrieval

**Goal:** store every chunk's embedding in a **FAISS** index that can be saved to disk and searched in milliseconds,
and fix the course-code weakness found in Stage 3 by adding **BM25 keyword search**.

**Architecture** (all in `src/vectorstore.py`, class `FaissVectorStore`)

```
 Stage 2 chunks ──► Stage 3 BGE embeddings (10,732 × 384, normalised)
        │                       │
        ▼                       ▼
 faiss_store/chunks.jsonl    faiss_store/index.faiss        faiss_store/manifest.json
 line i = chunk_id i:        IndexFlatIP: exact search,     model, dim, chunk size/overlap,
 source, file, page, text    inner product = cosine         chunk count, fingerprints, checksum
        │                       │                                  │
        └──────── load(): verify everything matches ◄──────────────┘
                         │
                         ├─► BM25 keyword index (rebuilt in memory, ~2-3 s)
                         └─► course-code map: "haws433" -> chunks mentioning HAWS 433

 question ─► names a course code that exists in the yearbooks?
               ├─ yes ─► code-aware hybrid: vector + BM25 ranked ONLY among chunks containing that code,
               │         combined with Reciprocal Rank Fusion (RRF)
               └─ no ──► vector search (BGE) only
```

**Key decisions**
1. **`IndexFlatIP` (exact search).** With 10,732 vectors, comparing the question against every chunk takes a few
   milliseconds, so there is no need for an approximate index (which could miss results).
2. **Metadata travels with every vector.** Row *i* of the index is chunk *i*; `chunks.jsonl` stores its `chunk_id`,
   `source`, `file_name`, `page` and full `text`, so every search result can be cited.
3. **Integrity checks (manifest + fingerprints).** The manifest records the embedding model, vector size, chunk
   size/overlap, chunk count, a **corpus fingerprint** (SHA-256 of every chunk's id, source, page and text, in
   order), a **build fingerprint** (corpus + embedding configuration) and a **checksum of the vectors**. `load()`
   refuses an index whose files disagree with each other, with the configured model/chunking, or (when given)
   with the chunks produced from the PDFs today — so stale or corrupted vectors can never be used silently.
4. **BM25 keyword search** (`rank-bm25`). Words are lower-cased and stopwords dropped; each course code also
   becomes one joined token, so `HAWS 433`, `HAWS433` and `haws 433` all match each other.
5. **Routed retrieval (the default).** Course-code questions use code-aware hybrid search; every other question
   uses BGE vector search only. Section 4.8 shows the evaluation behind this choice.

### 4.1 Build (once) and load the index

Embedding all chunks takes ~33 minutes on this CPU, so the index is built once and saved. The cell below only
builds it if `faiss_store/` does not exist yet; normally it just **loads and verifies** the saved index.
`expected_fingerprint` is the fingerprint of the chunks produced in Stage 2 *in this session*: if the PDFs or the
chunking had changed since the index was built, loading would fail with a "rebuild it" message.

In [20]:
from src.vectorstore import FaissVectorStore, chunk_to_record, corpus_fingerprint, MANIFEST_FILE

STORE_DIR = "faiss_store"

# Fingerprint of the chunks we just made in Stage 2 (same PDFs + same chunking -> same fingerprint)
current_fingerprint = corpus_fingerprint([chunk_to_record(c) for c in chunks])

store = FaissVectorStore(STORE_DIR)          # BGE model, chunk size 800 / overlap 150 by default
if not os.path.exists(os.path.join(STORE_DIR, MANIFEST_FILE)):
    print("No saved index: embedding all chunks (this takes ~30+ minutes) ...")
    store.build(chunks, embedder.embed_chunks(chunks))
    store.save()

start = time.time()
store.load(expected_fingerprint=current_fingerprint)   # raises ValueError if anything does not match
print(f"Loaded + verified in {time.time() - start:.1f} s")

[INFO] Loaded and verified 10732 vectors + chunk records from faiss_store
Loaded + verified in 3.6 s


In [21]:
# What the manifest records about this index
import json
for key in ["embedding_model", "embedding_dim", "normalized_embeddings", "chunk_size", "chunk_overlap",
            "chunk_count", "chunks_per_source", "corpus_fingerprint", "build_fingerprint",
            "vectors_checksum", "index_type", "faiss_version", "bm25", "created_at"]:
    print(f"{key:22} {json.dumps(store.manifest[key])}")

embedding_model        "BAAI/bge-small-en-v1.5"
embedding_dim          384
normalized_embeddings  true
chunk_size             800
chunk_overlap          150
chunk_count            10732
chunks_per_source      {"MSU Humanities Yearbook": 5614, "MSU Sciences Yearbook Vol. 2": 5118}
corpus_fingerprint     "sha256:545d0dabe1fbe022baf9dd5dfaadaaee6637490e9a2f5fef9e04cf1d455e41f5"
build_fingerprint      "sha256:8e0e357fc8e85e7b133d98ce33b80986846fb7d883ecc39572445d31a0299fa4"
vectors_checksum       "sha256:38d4153f6e6ef7315acf29185e7b389a33f383fd1d4b1588b86621cd6d20b22a"
index_type             "IndexFlatIP"
faiss_version          "1.15.1"
bm25                   {"variant": "BM25Okapi", "k1": 1.5, "b": 0.75, "tokenizer": "v1"}
created_at             "2026-10-06T11:09:35+00:00"


### 4.2 Integrity tests

**Before the Stage 3 vectors were reused** they were checked against the current chunks, not just a sample:
- the corpus was loaded and chunked twice in separate runs and gave the **same fingerprint** both times
  (`sha256:545d0dab…41f5`), so the fingerprint is deterministic;
- all 10,732 chunks were re-embedded and compared **row by row** with the Stage 3 vectors: every row was
  identical (largest difference 0.0). The index was built from the fresh vectors, produced in the same run that
  computed the fingerprint.

The cells below repeat the checks that can be run quickly.

In [22]:
# Test 1: one chunk record per vector, each with its citation metadata
n_vectors, n_records = store.index.ntotal, len(store.records)
print("FAISS vectors:", n_vectors, "| chunk records:", n_records, "| manifest chunk_count:", store.manifest["chunk_count"])
assert n_vectors == n_records == store.manifest["chunk_count"] == len(chunks)

# Row i must be chunk i, and every record keeps source, page, chunk_id and text
assert all(r["chunk_id"] == i for i, r in enumerate(store.records))
assert all(r["source"] and r["page"] and r["text"] for r in store.records)
assert all(r["text"] == c.page_content for r, c in zip(store.records, chunks))
print("Every vector has its chunk_id, source, page and text ✅")

# The fingerprint is deterministic: re-chunking the same pages gives the same value
assert corpus_fingerprint([chunk_to_record(c) for c in chunk_documents(docs)]) == store.manifest["corpus_fingerprint"]
print("Corpus fingerprint matches the manifest ✅")

FAISS vectors: 10732 | chunk records: 10732 | manifest chunk_count: 10732
Every vector has its chunk_id, source, page and text ✅


[INFO] Split 1956 documents into 10732 chunks.
Corpus fingerprint matches the manifest ✅


In [23]:
# Test 2: save to a new folder, reload, and check nothing changed
import shutil, tempfile

with tempfile.TemporaryDirectory() as tmp:
    copy_dir = os.path.join(tmp, "store_copy")
    store.persist_dir = copy_dir
    store.save()
    store.persist_dir = STORE_DIR

    reloaded = FaissVectorStore(copy_dir)
    reloaded.load(expected_fingerprint=current_fingerprint)
    same_vectors = np.array_equal(store.index.reconstruct_n(0, n_vectors), reloaded.index.reconstruct_n(0, n_vectors))
    same_records = reloaded.records == store.records
    print("Vectors identical after reload:", same_vectors, "| records identical:", same_records)
    assert same_vectors and same_records

    # Tamper tests: each damaged copy of the index must be refused when loading
    path = os.path.join(copy_dir, "chunks.jsonl")
    with open(path, encoding="utf-8") as f:
        original_lines = f.readlines()

    def damaged(edit):
        lines = list(original_lines)
        record = json.loads(lines[100])
        edit(record)
        lines[100] = json.dumps(record, ensure_ascii=False) + "\n"
        return lines

    tamper_cases = {
        "one character of a chunk's text changed": damaged(lambda r: r.update(text=r["text"].replace("e", "E", 1))),
        "a chunk's page number changed": damaged(lambda r: r.update(page=r["page"] + 1)),
        "a chunk's 'source' field removed": damaged(lambda r: r.pop("source")),
        "two chunks swapped": original_lines[:100] + [original_lines[101], original_lines[100]] + original_lines[102:],
    }
    for name, lines in tamper_cases.items():
        with open(path, "w", encoding="utf-8") as f:
            f.writelines(lines)
        try:
            FaissVectorStore(copy_dir).load()
            print(f"❌ {name}: tampered index was accepted")
        except ValueError as error:
            print(f"✅ {name}: rejected -> {str(error).splitlines()[-1].strip()}")

[INFO] Saved index, 10732 chunk records and manifest to C:\Users\donal\AppData\Local\Temp\tmpu04awdgf\store_copy


[INFO] Loaded and verified 10732 vectors + chunk records from C:\Users\donal\AppData\Local\Temp\tmpu04awdgf\store_copy
Vectors identical after reload: True | records identical: True


✅ one character of a chunk's text changed: rejected -> - chunk records do not match the manifest's corpus fingerprint


✅ a chunk's page number changed: rejected -> - chunk records do not match the manifest's corpus fingerprint


✅ a chunk's 'source' field removed: rejected -> - 1 chunk records lack ['chunk_id', 'file_name', 'page', 'source', 'text'] (first: line 101)


✅ two chunks swapped: rejected -> - chunk records do not match the manifest's corpus fingerprint


### 4.3 Size and speed

In [24]:
for name in ["index.faiss", "chunks.jsonl", "manifest.json"]:
    print(f"{name:14} {os.path.getsize(os.path.join(STORE_DIR, name)) / 1e6:6.2f} MB")
print("Load timings (s):", {k: round(v, 2) for k, v in store.timings.items()})

# Query latency: embed the question + search (the first call also loads the BGE model)
start = time.time(); store.query("warm-up"); print(f"First query (loads model): {time.time() - start:.1f} s")

import statistics
latency_questions = ["How do I defer my studies?", "What is HAWS 433 about?", "Who appoints the Vice-Chancellor?",
                     "Describe the module MPG 704", "Can I write a supplementary exam if I fail?"]
for mode in ["vector", "bm25", "routed"]:
    times = []
    for _ in range(3):
        for q in latency_questions:
            start = time.perf_counter(); store.query(q, top_k=5, mode=mode); times.append(time.perf_counter() - start)
    print(f"{mode:7} median {statistics.median(times) * 1000:6.1f} ms per question")

index.faiss     16.48 MB
chunks.jsonl     8.52 MB
manifest.json    0.00 MB
Load timings (s): {'read_s': 0.15, 'verify_s': 0.36, 'bm25_build_s': 3.06, 'save_s': 0.3}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  47%|████▋     | 93/199 [00:00<00:00, 910.74it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1948.78it/s]

[INFO] Loaded embedding model: BAAI/bge-small-en-v1.5 (dim=384)
First query (loads model): 7.3 s


vector  median   47.8 ms per question


bm25    median   19.1 ms per question


routed  median   66.6 ms per question


### 4.4 How BM25 sees text, and how questions are routed

Course codes become a single token (`haws433`) however they are written. Only codes that really appear in the
yearbooks (written in capitals there) trigger the course-code route, so phrases like "credit load 270" do not.

In [25]:
from src.vectorstore import bm25_tokenize

print(bm25_tokenize("What is HAWS433 about?"))
print(bm25_tokenize("Tell me about haws 433"))
print()
for question in ["What is HAWS 433 about?", "mpg 704", "What is the minimum credit load 270?",
                 "What is XYZ 999?", "How do I defer my studies?"]:
    ids = store.chunks_with_codes(question)
    route = f"code-aware hybrid over {len(ids)} chunks containing the code" if ids else "vector search"
    print(f"{question!r:42} -> {route}")

['haws', '433', 'haws433']
['haws', '433', 'haws433']

'What is HAWS 433 about?'                  -> code-aware hybrid over 7 chunks containing the code
'mpg 704'                                  -> code-aware hybrid over 2 chunks containing the code
'What is the minimum credit load 270?'     -> vector search
'What is XYZ 999?'                         -> vector search
'How do I defer my studies?'               -> vector search


### 4.5 Evaluation method

Five retrieval methods were compared on **20 questions**, each with a pattern that a correct chunk must match:
- the **10 Stage 3 questions** (same patterns as in Stage 3), and
- **10 course-code questions** written in different ways (`HAWS433`, `mpg 704`, `Describe EC439`, a bare `HFEE212`,
  codes from both books, codes that appear in up to 56 chunks). For these, only the module's **description**
  counts as correct — not a programme-structure table that merely lists the code.

| Method | What it does |
|---|---|
| `vector` | BGE embeddings + FAISS only (Stage 3 behaviour) |
| `bm25` | keyword search only |
| `rrf` | plain hybrid: vector top-50 and BM25 top-50 combined with RRF, score = Σ 1/(60 + rank) |
| `hybrid` | code-aware hybrid for **every** question (code questions restricted to chunks containing the code) |
| `routed` | **chosen**: code-aware hybrid for code questions, vector only for everything else |

Metrics: **top-1** (correct chunk ranked first), **top-3** (in the first three), **top-5** (correct chunk anywhere
in the 5 passages the LLM will read) and **MRR@5** (average of 1/rank, 0 if not in the top 5). The top results were also checked by hand,
because some Stage 3 patterns are lenient (e.g. any chunk containing "defer").

In [26]:
import re
import pandas as pd

# (question, regex a correct chunk must match)
EVAL_QUESTIONS = [
    ("What is HAWS 433 about?", r"HAWS 433"),
    ("Describe the module MPG 704", r"MPG 704 Research Methods"),
    ("How do I defer my studies?", r"(?i)defer"),
    ("How are bachelor's degrees classified?", r"(?i)(degree classification|first class)"),
    ("Who appoints the Vice-Chancellor?", r"(?i)Vice-Chancellor shall be appointed"),
    ("What does the Student Disciplinary Committee do?", r"(?i)Disciplinary Committee"),
    ("What happens if a student is caught plagiarising?", r"(?i)plagiari"),
    ("Can I write a supplementary exam if I fail?", r"(?i)supplementary"),
    ("What is voluntary termination of studies?", r"(?i)voluntary termination"),
    ("Which module covers statistics for forensic science?", r"(?i)Statistics for Forensic"),
    # Course-code questions: correct = the module DESCRIPTION
    ("What is HAWS433 about?", r"HAWS ?433 Biotechnology in Animal Science\nMethods"),
    ("mpg 704", r"MPG 704 Research Methods in Gender, Peace and Security\nThis module"),
    ("What is BSS 204 about?", r"BSS ?204 World Political Systems\nThe module"),
    ("What does HRFS 412 cover?", r"HRFS ?412 New Testament Ethics\nThis module"),
    ("Tell me about PSY 134", r"PSY ?134 Human Diversity\nThe module"),
    ("What is taught in HMPC 433?", r"HMPC ?433 Organic Chemistry III\nThis module"),
    ("Describe EC439", r"EC ?439 Development Economics\nThe module"),
    ("What is BED133 about?", r"BED ?133:? [^\n]+\nThis module"),  # BED133 names different modules in different programmes
    ("HFEE212", r"HFEE ?212 Fluid Mechanics\nThis module"),
    ("What is the code for the module Human Evolution?", r"ANTH ?102 Human Evolution\nThis module"),
]
MODES = ["vector", "bm25", "rrf", "hybrid", "routed"]

def correct_rank(results, pattern):
    # Rank (1-5) of the first correct chunk, or None if no correct chunk is in the top 5
    return next((r["rank"] for r in results if re.search(pattern, r["text"])), None)

eval_results = {}   # (question number, mode) -> list of results, reused below
rows = []
for number, (question, pattern) in enumerate(EVAL_QUESTIONS, 1):
    row = {"#": number, "question": question}
    for mode in MODES:
        eval_results[number, mode] = store.query(question, top_k=5, mode=mode)
        row[mode] = correct_rank(eval_results[number, mode], pattern)
    rows.append(row)

ranks = pd.DataFrame(rows).set_index("#")
ranks[MODES] = ranks[MODES].astype("Int64")            # whole numbers, <NA> = not in the top 5
ranks.astype(object).fillna("-")   # rank of the correct chunk per method ('-' = not in the top 5)

,question,vector,bm25,rrf,hybrid,routed
#,,,,,,
1,What is HAWS 433 about?,4,1,1,1,1
2,Describe the module MPG 704,-,1,4,1,1
3,How do I defer my studies?,1,1,1,1,1
4,How are bachelor's degrees classified?,1,1,1,1,1
5,Who appoints the Vice-Chancellor?,1,3,1,1,1
6,What does the Student Disciplinary Committee do?,1,1,1,1,1
7,What happens if a student is caught plagiarising?,1,-,-,-,1
8,Can I write a supplementary exam if I fail?,1,1,1,1,1
9,What is voluntary termination of studies?,2,3,2,2,2


In [27]:
def summarise(table):
    return pd.DataFrame({
        mode: {"top-1": f"{(table[mode] == 1).sum()}/{len(table)}",
               "top-3": f"{(table[mode] <= 3).sum()}/{len(table)}",
               "top-5": f"{table[mode].notna().sum()}/{len(table)}",
               "MRR@5": round((1 / table[mode]).fillna(0).mean(), 2)}
        for mode in MODES}).T

print("Stage 3 questions (1-10)");  display(summarise(ranks.loc[1:10]))
print("Course-code questions (11-20)"); display(summarise(ranks.loc[11:20]))
print("All 20 questions"); display(summarise(ranks))

# Did any method push down an answer that vector search already ranked correctly?
for mode in ["rrf", "hybrid", "routed"]:
    worse = [n for n in ranks.index if pd.notna(ranks.loc[n, "vector"])
             and (pd.isna(ranks.loc[n, mode]) or ranks.loc[n, mode] > ranks.loc[n, "vector"])]
    print(f"{mode:7} ranks the correct chunk lower than vector search on questions: {worse or 'none'}")

Stage 3 questions (1-10)


,top-1,top-3,top-5,MRR@5
vector,6/10,8/10,9/10,0.71
bm25,7/10,9/10,9/10,0.77
rrf,7/10,8/10,9/10,0.78
hybrid,8/10,9/10,9/10,0.85
routed,8/10,10/10,10/10,0.88


Course-code questions (11-20)


,top-1,top-3,top-5,MRR@5
vector,1/10,1/10,2/10,0.12
bm25,7/10,9/10,10/10,0.82
rrf,3/10,6/10,6/10,0.42
hybrid,6/10,9/10,10/10,0.76
routed,6/10,9/10,10/10,0.76


All 20 questions


,top-1,top-3,top-5,MRR@5
vector,7/20,9/20,11/20,0.41
bm25,14/20,18/20,19/20,0.8
rrf,10/20,14/20,15/20,0.6
hybrid,14/20,18/20,19/20,0.8
routed,14/20,19/20,20/20,0.82


rrf     ranks the correct chunk lower than vector search on questions: [7]
hybrid  ranks the correct chunk lower than vector search on questions: [7]
routed  ranks the correct chunk lower than vector search on questions: none


### 4.6 Top results with source and page

`show()` prints the top results of one method for one question: `*` marks a chunk matching the "correct" pattern,
and for hybrid results `v#` / `b#` show the rank each retriever gave that chunk.

In [28]:
def show(number, mode, top=3):
    question = EVAL_QUESTIONS[number - 1][0]
    pattern = EVAL_QUESTIONS[number - 1][1]
    print(f"Q{number} [{mode}] {question}")
    for r in eval_results[number, mode][:top]:
        mark = "*" if re.search(pattern, r["text"]) else " "
        ranks_used = f" (v#{r['vector_rank'] or '-'}, b#{r['bm25_rank'] or '-'})" if "vector_rank" in r else ""
        body = " ".join(r["text"].split("\n", 1)[1].split())[:95]   # text after the citation header
        print(f"  {mark}{r['rank']}. {r['source']}, p.{r['page']} [chunk {r['chunk_id']}]{ranks_used}  {body}")
    print()

# The course-code problem: vector search cannot find the module, routed search can
for mode in ["vector", "routed"]:
    show(1, mode)
for mode in ["vector", "routed"]:
    show(14, mode)

Q1 [vector] What is HAWS 433 about?
   1. MSU Sciences Yearbook Vol. 2, p.284 [chunk 6477]  HAWS 439 Research Project Explore research issues, research area selection, research concept de
   2. MSU Sciences Yearbook Vol. 2, p.300 [chunk 6564]  HAWS 439 Research Project Explore research issues, research area selection, research concept de
   3. MSU Sciences Yearbook Vol. 2, p.308 [chunk 6604]  HAWS 439 Research Project Explore research issues, research area selection, research concept de

Q1 [routed] What is HAWS 433 about?
  *1. MSU Sciences Yearbook Vol. 2, p.283 [chunk 6468] (v#2, b#1)  HAWS 433 Biotechnology in Animal Science Methods in Biotechnology- traditional and modern: Ethi
  *2. MSU Sciences Yearbook Vol. 2, p.296 [chunk 6536] (v#1, b#3)  HAWS233 12 HAWS 443 Replacement Animal Rearing and Management 12 HAWS 433 Biotechnology in Anim
  *3. MSU Sciences Yearbook Vol. 2, p.289 [chunk 6493] (v#6, b#2)  Level 4 Semester 1 Code Module Description Prerequisite Credits HAWS 430 Meat 

In [29]:
# Every question with the chosen (routed) method
for number in range(1, len(EVAL_QUESTIONS) + 1):
    show(number, "routed", top=3)

Q1 [routed] What is HAWS 433 about?
  *1. MSU Sciences Yearbook Vol. 2, p.283 [chunk 6468] (v#2, b#1)  HAWS 433 Biotechnology in Animal Science Methods in Biotechnology- traditional and modern: Ethi
  *2. MSU Sciences Yearbook Vol. 2, p.296 [chunk 6536] (v#1, b#3)  HAWS233 12 HAWS 443 Replacement Animal Rearing and Management 12 HAWS 433 Biotechnology in Anim
  *3. MSU Sciences Yearbook Vol. 2, p.289 [chunk 6493] (v#6, b#2)  Level 4 Semester 1 Code Module Description Prerequisite Credits HAWS 430 Meat Technology 12 HAW

Q2 [routed] Describe the module MPG 704
  *1. MSU Humanities Yearbook, p.885 [chunk 5301] (v#2, b#1)  Written Examinations: End-of-semester examinations constitute 60% of the final mark. 9.0 PROVIS
  *2. MSU Humanities Yearbook, p.886 [chunk 5305] (v#1, b#2)  in the international relations arena. The United Nations Security Council resolution 1325 recog

Q3 [routed] How do I defer my studies?
  *1. MSU Sciences Yearbook Vol. 2, p.156 [chunk 5834]  11. Deferment of Studi

### 4.7 Why plain RRF / hybrid-for-everything was not used

The two questions where fusing BM25 into *every* search lowered the correct answer below vector search:

In [30]:
# Q7: BM25 has no stemming, so "plagiarising" never matches "plagiarism" (the only form in the yearbooks).
# Its unrelated results ("student", "caught") are fused in and push all five correct chunks out of the top 5.
show(7, "vector"); show(7, "bm25"); show(7, "hybrid")

# Q3: the pattern accepts any chunk mentioning deferment, but by hand: hybrid's #1 is about refunds of fees,
# while vector search puts the actual "Deferment of Studies" regulation first.
show(3, "vector"); show(3, "hybrid")

Q7 [vector] What happens if a student is caught plagiarising?
  *1. MSU Humanities Yearbook, p.146 [chunk 615]  19.6.2 In the case of plagiarism being discovered in a project for the second time and after re
  *2. MSU Humanities Yearbook, p.123 [chunk 500]  12.5 Minor Cases of Plagiarism • First Offence: in the case of plagiarism being discovered in a
  *3. MSU Humanities Yearbook, p.138 [chunk 583]  18.5 Minor Cases of Plagiarism • First Offence: in the case of plagiarism being discovered in a

Q7 [bm25] What happens if a student is caught plagiarising?
   1. MSU Humanities Yearbook, p.522 [chunk 2971]  creoles. Implications for the classroom in terms of selection and grading of content, teaching 
   2. MSU Humanities Yearbook, p.179 [chunk 796]  the core values of heritage, identifies cultural heritage management processes and practices, t
   3. MSU Humanities Yearbook, p.107 [chunk 432]  4.2.4 to recommend to the Registrar that a student be summoned to appear before the Student Dis


### 4.8 Results and decision

Top-1 / top-3 / top-5:

| Method | Stage 3 questions (1-10) | Course-code questions (11-20) | All 20 | Lowers a correct vector answer? |
|---|---|---|---|---|
| vector | 6 · 8 · 9 /10 | 1 · 1 · 2 /10 | 7 · 9 · 11 /20 | — |
| bm25 | 7 · 9 · 9 /10 | 7 · 9 · 10 /10 | 14 · 18 · 19 /20 | yes (Q5, Q7, Q9) |
| rrf (plain hybrid) | 7 · 8 · 9 /10 | 3 · 6 · 6 /10 | 10 · 14 · 15 /20 | yes (Q7, and Q3 by hand) |
| hybrid (code-aware, every question) | 8 · 9 · 9 /10 | 6 · 9 · 10 /10 | 14 · 18 · 19 /20 | yes (Q7, and Q3 by hand) |
| **routed** (chosen) | **8 · 10 · 10 /10** | **6 · 9 · 10 /10** | **14 · 19 · 20 /20** | **no** |

**How to read this honestly**
- Routed is not more accurate at rank 1: it ties BM25 and code-aware hybrid (14/20). It is the most **reliable**
  method: best top-3 and top-5, and the only one that never ranks a correct answer lower than vector search.
- Most of the gap over vector comes from the course-code questions, which were added to test the known weakness.
  On natural-language questions routed *is* vector search, so this set can only show it does no harm there.
- These 20 questions were used to make design decisions (routing, the BM25-first tie-break, the BED133 check), so
  they are no longer an unbiased test. Stage 5 evaluates the frozen system on new, unseen questions.

**Why routed retrieval**
- **Plain RRF does not fix course codes** (top-5 only 6/10): with equal weighting, a chunk that both retrievers
  rank moderately beats BM25's exact match, and for a code question the vector list is mostly other modules.
  Restricting both retrievers to the chunks that contain the code (code-aware hybrid) puts the right module's
  description in the top 5 for **10/10** code questions (vector search alone: 2/10).
- **Fusing BM25 into natural-language questions can hurt.** On Q7 it removed every correct chunk from the top 5,
  and on Q3 it replaced the deferment regulation with a fee-refund clause. BGE already handles these questions
  well, so routed retrieval leaves them to vector search: **no question got worse than with vector search alone**.
- **The trade-off:** BM25 fusion did help two natural-language questions (Q10 forensic statistics, rank 3 → 1;
  Q6 by hand). Routed retrieval gives that up; the correct chunk is still in the top 5 the LLM will read.

**Known limitations**
- For 4 course-code questions the description is ranked 2nd–4th behind a programme table listing the same code;
  it is still within the 5 passages given to the LLM.
- BM25 has no stemming (`plagiarising` ≠ `plagiarism`); this only matters for code questions now.
- The evaluation set is small (20 questions) and Stage 5 re-tests retrieval on unseen questions.

**Stage 4 result:** a 25 MB index (`faiss_store/`: 10,732 BGE vectors + chunk records with source, page,
chunk_id and text + manifest) that loads and verifies itself in ~3 s. Retrieval is routed: course-code questions
use code-aware hybrid search (vector + BM25 + RRF within the chunks containing the code), all other questions use
BGE vector search. Correct passage in the top 5 for 20/20 test questions (vector only: 11/20), with no question
answered worse than by vector search.

## Stage 5 — Unseen retrieval evaluation

**Goal:** measure how well the *frozen* Stage 4 system retrieves on questions it was never designed or tuned on,
and find out **why** it fails — before changing anything. The Stage 4 questions were used to make design decisions,
so they cannot show how the system performs on new questions.

**Rules (agreed before any question was run)**
1. 30 new questions + 5 out-of-scope questions, written and approved, then **frozen** (file hash recorded) together
   with an answer key: what a sufficient answer must contain and every acceptable source page.
2. The retrieval system is not changed during Stage 5. Its files are checked against the Stage 4 freeze hashes.
3. Every retrieved chunk is **graded blind**: chunks from all methods are pooled per question and shuffled, and the
   grader sees only the chunk text — not which method found it or at what rank.
   **2** = sufficient on its own for an LLM to answer correctly · **1** = relevant but not sufficient · **0** = irrelevant.
4. A method "finds" the answer at rank *k* if a grade-2 chunk is at rank ≤ *k*. Scored with `top_k = 5`
   (top-1 / top-3 / top-5). Ranks 6–10 are logged and graded **only for diagnosis** of misses.
5. Every miss gets a recorded failure reason.
6. **Decision rule (fixed in advance):** routed top-5 ≥ 27/30 with top-1 ≤ 21/30 → ranking problem, investigate
   reranking · routed top-5 < 24/30 → retrieval problem, investigate retrieval/chunking/routing first ·
   24–26 → inconclusive, use the failure categories · top-5 ≥ 27 with top-1 > 21 → no reranker needed.

**How the questions were chosen** (to avoid picking easy or hard ones by hand)
- **Course codes:** a seeded random draw (seed 20261007) from module-description chunks in each yearbook, excluding
  Stage 4 codes; 10 for exact-code questions, 10 for code-in-context questions. Four code-in-context questions ask
  for programme-table facts (credits, semester, other modules), where the *table* is the correct chunk.
- **Natural-language topics:** a seeded random draw from section headings of the general regulations, taken in
  sample order with fixed skip rules (department/faculty names, at most two postgraduate-research topics, headings
  without substantive text).
- **Answer pages** were found by plain text search of the yearbooks, not by any retriever.

### 5.1 Verify the freeze

In [31]:
import hashlib

STAGE5_QUESTIONS_SHA256 = "434045f9a63ad2ba79aba6fd63e031f8c96564179025db9154d2531839817f0d"  # recorded at freeze

raw = open("eval/stage5_questions.json", "rb").read()
assert hashlib.sha256(raw).hexdigest() == STAGE5_QUESTIONS_SHA256, "frozen question file has changed!"
frozen = json.loads(raw)
print("Questions frozen at:", frozen["frozen_at"], "| sha256 matches ✅")

# The retrieval system must be exactly the frozen Stage 4 system (the notebook itself is allowed to grow)
for path, digest in frozen["stage4_freeze"]["sha256"].items():
    if path.endswith(".ipynb"):
        continue
    assert hashlib.sha256(open(path, "rb").read()).hexdigest() == digest, f"Stage 4 file changed: {path}"
print("Stage 4 system files unchanged since the freeze ✅")

questions = frozen["questions"]
print(collections.Counter(q["category"] for q in questions))

Questions frozen at: 2026-10-07T10:14:12+02:00 | sha256 matches ✅
Stage 4 system files unchanged since the freeze ✅
Counter({'natural_language': 10, 'exact_code': 10, 'code_in_context': 10, 'out_of_scope': 5})


In [32]:
# The frozen question set and answer key
pd.set_option("display.max_colwidth", 90)
pd.DataFrame([{"id": q["id"], "category": q["category"], "question": q["question"],
               "acceptable pages": ", ".join(q.get("pages", [])) or "(out of scope)"} for q in questions]).set_index("id")

,category,question,acceptable pages
id,,,
N1,natural_language,What is a faculty board responsible for?,"Hum 31, Sci 32, Hum 81"
N2,natural_language,How are archaeology students taught and assessed?,Hum 169
N3,natural_language,Who chairs Council meetings when the Chancellor is not there?,"Hum 18, Sci 19"
N4,natural_language,Who are the members of Midlands State University?,"Hum 14, Hum 15"
N5,natural_language,"Can I switch from an MPhil to a PhD, and when?","Hum 151, Sci 154"
N6,natural_language,"If something in my residence room gets broken, will I have to pay for it?",Hum 108
N7,natural_language,What qualification do I need to apply for an MPhil?,"Hum 149, Sci 151"
N8,natural_language,What do I have to sign when I register as a student?,Hum 105
N9,natural_language,How does the university decide to award an honorary doctorate?,"Hum 156, Sci 159"


### 5.2 Run the five methods (and check the stored run is reproducible)

All methods were run once with `top_k = 10` (ranks 1–5 scored, 6–10 diagnosis only); a separate `top_k = 5` run was
asserted identical to the first five. The run was saved to `eval/stage5_runs.json` **before grading**. The cell below
runs everything again and checks that the results are identical, so the graded results really belong to this system.

In [33]:
stored = json.load(open("eval/stage5_runs.json"))
STAGE5_MODES = stored["modes"]                                   # vector, bm25, rrf, hybrid, routed
runs = stored["runs"]

mismatches = 0
for q in questions:
    for mode in STAGE5_MODES:
        now = [r["chunk_id"] for r in store.query(q["question"], top_k=10, mode=mode)]
        mismatches += now != [r["chunk_id"] for r in runs[q["id"]][mode]]
print(f"Re-ran {len(questions)} questions x {len(STAGE5_MODES)} methods: {mismatches} differences from the stored run")
assert mismatches == 0

Re-ran 35 questions x 5 methods: 0 differences from the stored run


### 5.3 Blind grades

`eval/stage5_grades.json` holds one grade per (question, chunk): 289 chunks from the pooled top-5 results of all
methods, plus 84 chunks at ranks 6–10 graded afterwards (blind, the same way) only for the 34 method/question pairs
with no sufficient chunk in their top 5.

In [34]:
grade_rows = json.load(open("eval/stage5_grades.json"))["grades"]
grade = {(g["question_id"], g["chunk_id"]): g["grade"] for g in grade_rows}

scored = [q for q in questions if q["category"] != "out_of_scope"]
ungraded = [(q["id"], m) for q in scored for m in STAGE5_MODES for r in runs[q["id"]][m][:5]
            if (q["id"], r["chunk_id"]) not in grade]
print("Chunks graded:", len(grade_rows), dict(collections.Counter(g["phase"] for g in grade_rows)))
print("Top-5 results without a grade:", len(ungraded))
assert not ungraded
print("Grade distribution (top-5 pool):",
      dict(sorted(collections.Counter(g["grade"] for g in grade_rows if g["phase"] == "top5").items())))

Chunks graded: 373 {'top5': 289, 'rank6_10': 84}
Top-5 results without a grade: 0
Grade distribution (top-5 pool): {0: 208, 1: 41, 2: 40}


### 5.4 Results

In [35]:
CATEGORIES = ["natural_language", "exact_code", "code_in_context"]

def first_sufficient(qid, mode):
    # Rank (1-10) of the first grade-2 chunk, or None
    return next((r["rank"] for r in runs[qid][mode] if grade.get((qid, r["chunk_id"])) == 2), None)

first = {(q["id"], m): first_sufficient(q["id"], m) for q in scored for m in STAGE5_MODES}

def results_table(k):
    table = {}
    for mode in STAGE5_MODES:
        row = {}
        for cat in CATEGORIES + ["overall"]:
            ids = [q["id"] for q in scored if cat == "overall" or q["category"] == cat]
            hits = sum(1 for i in ids if first[i, mode] is not None and first[i, mode] <= k)
            row[cat] = f"{hits}/{len(ids)}"
        table[mode] = row
    return pd.DataFrame(table).T

for k in (1, 3, 5):
    print(f"Top-{k}: a sufficient chunk within the first {k} results")
    display(results_table(k))

Top-1: a sufficient chunk within the first 1 results


,natural_language,exact_code,code_in_context,overall
vector,6/10,4/10,4/10,14/30
bm25,3/10,3/10,6/10,12/30
rrf,4/10,4/10,6/10,14/30
hybrid,4/10,3/10,6/10,13/30
routed,6/10,3/10,6/10,15/30


Top-3: a sufficient chunk within the first 3 results


,natural_language,exact_code,code_in_context,overall
vector,9/10,4/10,4/10,17/30
bm25,5/10,10/10,9/10,24/30
rrf,5/10,6/10,6/10,17/30
hybrid,5/10,10/10,9/10,24/30
routed,9/10,10/10,9/10,28/30


Top-5: a sufficient chunk within the first 5 results


,natural_language,exact_code,code_in_context,overall
vector,9/10,4/10,5/10,18/30
bm25,5/10,10/10,9/10,24/30
rrf,7/10,7/10,6/10,20/30
hybrid,7/10,10/10,9/10,26/30
routed,9/10,10/10,9/10,28/30


In [36]:
# Rank of the first sufficient chunk per question (ranks 6-10 shown for diagnosis only; '-' = not in the top 10)
per_question = pd.DataFrame([{"id": q["id"], "category": q["category"],
                              **{m: first[q["id"], m] for m in STAGE5_MODES}} for q in scored]).set_index("id")
per_question[STAGE5_MODES] = per_question[STAGE5_MODES].astype("Int64")
per_question.astype(object).fillna("-")

,category,vector,bm25,rrf,hybrid,routed
id,,,,,,
N1,natural_language,2,2,2,2,2
N2,natural_language,1,-,-,-,1
N3,natural_language,1,1,1,1,1
N4,natural_language,-,-,-,-,-
N5,natural_language,1,3,1,1,1
N6,natural_language,1,-,5,5,1
N7,natural_language,3,-,-,-,3
N8,natural_language,1,1,1,1,1
N9,natural_language,1,-,5,5,1


In [37]:
# Does routed retrieval ever rank the answer LOWER than plain vector search? (Stage 4 found no such case)
lower = [q["id"] for q in scored if first[q["id"], "vector"] and first[q["id"], "vector"] <= 5
         and (first[q["id"], "routed"] is None or first[q["id"], "routed"] > first[q["id"], "vector"])]
higher = [q["id"] for q in scored if first[q["id"], "routed"] and first[q["id"], "routed"] <= 5
          and (first[q["id"], "vector"] is None or first[q["id"], "vector"] > first[q["id"], "routed"])]
print("Routed ranks the answer lower than vector on:", lower)
print("Routed ranks the answer higher than vector on:", higher)

Routed ranks the answer lower than vector on: ['E1', 'E6', 'E7', 'C3']
Routed ranks the answer higher than vector on: ['E2', 'E3', 'E4', 'E5', 'E8', 'E10', 'C1', 'C2', 'C6', 'C7', 'C9']


### 5.5 Why it fails

Every method/question pair without a sufficient chunk in its top 5 has a recorded reason
(`eval/stage5_failures.json`), assigned after inspecting the results and the ranks of the correct chunks.

In [38]:
failures = json.load(open("eval/stage5_failures.json"))
misses = pd.DataFrame(failures["top5_misses"])
assert len(misses) == sum(1 for q in scored for m in STAGE5_MODES if not (first[q["id"], m] and first[q["id"], m] <= 5))
for key, text in failures["reasons"].items():
    print(f"{key:28} {text}")
display(pd.crosstab(misses["reason"], misses["method"], margins=True))
misses.set_index(["question_id", "method"])

code_not_in_embedding        Vector embeddings do not represent course codes: retrieves look-alike modules or programme tables instead of the named module
unfiltered_fusion            Plain RRF has no code filter: chunks that the vector list ranks moderately (other modules) outrank BM25's exact code match
bm25_vocabulary_mismatch     BM25 needs the same words: question wording differs from the regulation's wording (no stemming/synonyms), so other chunks win
fusion_dilution              Fusing BM25 into a natural-language question pushes vector's correct chunk below chunks both lists rank moderately
generic_query_list_answer    Generic question ('members of the University'); the answer chunk is a bare list with few topical words, ranked ~100th by both retrievers
code_crowding                Code appears in 15 chunks (programme tables, 'refer to department' stubs); the 3 description chunks rank 8th-12th within that set


method,bm25,hybrid,routed,rrf,vector,All
reason,,,,,,
bm25_vocabulary_mismatch,4,0,0,0,0,4
code_crowding,1,1,1,1,1,5
code_not_in_embedding,0,0,0,0,10,10
fusion_dilution,0,2,0,2,0,4
generic_query_list_answer,1,1,1,1,1,5
unfiltered_fusion,0,0,0,6,0,6
All,6,4,2,10,12,34


reason  \
question_id method                              
N2          bm25     bm25_vocabulary_mismatch   
            rrf               fusion_dilution   
            hybrid            fusion_dilution   
N4          vector  generic_query_list_answer   
            bm25    generic_query_list_answer   
            rrf     generic_query_list_answer   
            hybrid  generic_query_list_answer   
            routed  generic_query_list_answer   
N6          bm25     bm25_vocabulary_mismatch   
N7          bm25     bm25_vocabulary_mismatch   
            rrf               fusion_dilution   
            hybrid            fusion_dilution   
N9          bm25     bm25_vocabulary_mismatch   
E2          vector      code_not_in_embedding   
            rrf             unfiltered_fusion   
E3          vector      code_not_in_embedding   
E4          vector      code_not_in_embedding   
            rrf             unfiltered_fusion   
E5          vector      code_not_in_embedding   
E8          vector      code_not_in_embedding   
E10         vector      code_not_in_embedding   
            rrf             unfiltered_fusion   
C1          vector      code_not_in_embedding   
            rrf             unfiltered_fusion   
C6          vector      code_not_in_embedding   
C7          vector      code_not_in_embedding   
            rrf             unfiltered_fusion   
C9          vector      code_not_in_embedding   
            rrf             unfiltered_fusion   
C10         vector              code_crowding   
            bm25                code_crowding   
            rrf                 code_crowding   
            hybrid              code_crowding   
            routed              code_crowding   

                                                                                                         note  
question_id method                                                                                             
N2          bm25    'taught/assessed' vs 'TEACHING AND LEARNING METHODS / ASSESSMENT METHODS'; archaeology...  
            rrf                                                                                                
            hybrid                                                                                             
N4          vector                                membership chunk (Hum p14) is vector rank 109, BM25 rank 71  
            bm25                                                                                               
            rrf                                                                                                
            hybrid                                                                                             
            routed                                                                                             
N6          bm25    'broken/pay' vs 'damage/compensate'; warden/proctor 'financial loss' clauses win (grad...  
N7          bm25    'qualification' vs 'Applicants must have obtained'; 'mphil' matches MPhil programme li...  
            rrf                                                                                                
            hybrid                                                                                             
N9          bm25    'honorary' matches short departmental 'honorary degrees will be awarded' notes; 'docto...  
E2          vector                                                                                             
            rrf                                                                                                
E3          vector                                                           sufficient chunk only at rank 10  
E4          vector                                                                                             
            rrf                                                                    sufficient chunk at rank 6  
E5          vector                

In [39]:
# Routed retrieval: why the sufficient chunk is not ranked first (15 of 30 questions)
pd.DataFrame(failures["routed_top1_losses"]).set_index("question_id")

,explanation
question_id,
N1,Adjacent clause first: 'Composition of Faculty Boards' chunk (grade 1) above the funct...
N4,Retrieval miss (see failures)
N7,"Adjacent clause first: PhD entry requirement (5.2) above the MPhil requirement (5.1, #3)"
N10,Related rule first: '14.6 Repeat' (who may repeat) above the marks rule (#2)
E1,"Programme table first; RRF tie (v#2,b#1 vs v#1,b#2) broken towards BM25 by the Stage 4..."
E6,Programme table first; RRF tie broken towards BM25 by the Stage 4 tie-break
E7,Programme table first; RRF tie broken towards BM25 by the Stage 4 tie-break
E10,Programme table first; RRF tie broken towards BM25 by the Stage 4 tie-break
C3,Programme table first; RRF tie broken towards BM25 by the Stage 4 tie-break


In [40]:
def show_graded(qid, mode, top=5):
    q = next(q for q in questions if q["id"] == qid)
    print(f"{qid} [{mode}] {q['question']}")
    for r in runs[qid][mode][:top]:
        body = " ".join(store.records[r["chunk_id"]]["text"].split("\n", 1)[1].split())[:85]
        used = f" (v#{r['vector_rank'] or '-'}, b#{r['bm25_rank'] or '-'})" if r.get("vector_rank") or r.get("bm25_rank") else ""
        print(f"  grade {grade.get((qid, r['chunk_id']), '?')} | {r['rank']}. {r['source']}, p.{r['page']}{used}  {body}")
    print()

show_graded("E1", "vector", 3)   # vector puts the description first ...
show_graded("E1", "routed", 3)   # ... routed puts the programme table first (RRF tie broken towards BM25)
show_graded("C10", "routed")     # code in 15 chunks: tables and 'refer to department' stubs fill the top 5
show_graded("N4", "routed")      # generic question: the membership list is never retrieved

E1 [vector] What is DSC 603 about?
  grade 2 | 1. MSU Humanities Yearbook, p.304  DSC 602 Public Relations Communication This module focuses on the theoretical and pra
  grade 0 | 2. MSU Humanities Yearbook, p.304  DSC 608 Introduction to Communication Technology The module provides an overview of n
  grade 0 | 3. MSU Humanities Yearbook, p.304  point of discussion. DSC 606 English for Specific Purposes The module educates studen

E1 [routed] What is DSC 603 about?
  grade 1 | 1. MSU Humanities Yearbook, p.303 (v#2, b#1)  12. DEGREE WEIGHTING Refer to Section 10 of the Faculty Regulations for Postgraduate 
  grade 2 | 2. MSU Humanities Yearbook, p.304 (v#1, b#2)  DSC 602 Public Relations Communication This module focuses on the theoretical and pra

C10 [routed] As a first-year accounting student I have to take BM135. What is it about?
  grade 1 | 1. MSU Sciences Yearbook Vol. 2, p.451 (v#1, b#1)  BM131 Principles of Management (Refer to Department of Management Sciences for the mo
  gr

### 5.6 Out-of-scope questions (recorded for later; not scored)

The similarity of the best chunk is recorded so a "not found in the documents" threshold can be studied later.

In [41]:
oos = pd.DataFrame([{"id": q["id"], "category": q["category"], "question": q["question"],
                     "top-1 cosine (vector)": round(runs[q["id"]]["vector"][0]["score"], 3),
                     "routed route": runs[q["id"]]["routed"][0]["route"]} for q in questions])
display(oos[oos["category"] == "out_of_scope"].set_index("id"))
print(oos.groupby("category")["top-1 cosine (vector)"].agg(["min", "median", "max"]))

,category,question,top-1 cosine (vector),routed route
id,,,,
O1,out_of_scope,What is on the menu in the student cafeteria today?,0.682,vector
O2,out_of_scope,Who won the 2023 Zimbabwe presidential election?,0.612,vector
O3,out_of_scope,What is the WiFi password for the MSU library?,0.725,vector
O4,out_of_scope,How do I apply for a UK student visa?,0.647,vector
O5,out_of_scope,What time does the campus gym open?,0.617,vector


                    min  median    max
category                              
code_in_context   0.671  0.7290  0.808
exact_code        0.574  0.6685  0.718
natural_language  0.700  0.7470  0.805
out_of_scope      0.612  0.6470  0.725


### 5.7 Results, analysis and decision

| Method | Natural language top-1 · 3 · 5 | Exact code top-1 · 3 · 5 | Code in context top-1 · 3 · 5 | **Overall top-1 · 3 · 5** |
|---|---|---|---|---|
| vector | 6 · 9 · 9 | 4 · 4 · 4 | 4 · 4 · 5 | 14 · 17 · 18 /30 |
| bm25 | 3 · 5 · 5 | 3 · 10 · 10 | 6 · 9 · 9 | 12 · 24 · 24 /30 |
| rrf | 4 · 5 · 7 | 4 · 6 · 7 | 6 · 6 · 6 | 14 · 17 · 20 /30 |
| hybrid | 4 · 5 · 7 | 3 · 10 · 10 | 6 · 9 · 9 | 13 · 24 · 26 /30 |
| **routed** | **6 · 9 · 9** | **3 · 10 · 10** | **6 · 9 · 9** | **15 · 28 · 28 /30** |

**What the evidence shows**
- **Retrieval mostly works.** Routed puts a sufficient passage in the top 5 for **28/30** unseen questions and the
  top 3 for 28/30 (vector alone: 18/30). The Stage 4 choice holds up: plain RRF (20/30) and hybrid-for-everything
  (26/30) are both worse, for the same reasons seen in Stage 4 (unfiltered fusion; fusion diluting vector answers).
- **Ranking is the weak point.** Routed ranks the sufficient passage **first for only 15/30**. 11 of its 15 top-1
  losses are course-code questions where a **programme table** (grade 1: code, title, credits) is ranked above the
  module **description**:
  - in 5 of them (E1, E6, E7, E10, C3) the table and description tie in RRF and the **Stage 4 tie-break (BM25
    wins in code mode) picks the table** — this rule was chosen on two Stage 4 questions and does not generalise;
  - in 5 (E2, E4, C1, C7, C10) *both* retrievers rank the table first;
  - in 1 (E3) the code names **two different modules** in different programmes (SEMA204 is "Real Analysis" in one
    and "Introduction to Numerical Methods" in another) — a genuinely ambiguous question.
  The other 4 losses are natural-language questions where an adjacent clause is ranked first (N1, N7, N10) or
  retrieval failed (N4).
- **Stage 4's "never worse than vector" no longer holds at rank level.** On E1, E6, E7 and C3 vector search ranks
  the description first and routed ranks it second (still in the top 5).
- **Two retrieval failures a reranker over the top 5 cannot fix:**
  - **N4** ("Who are the members of MSU?"): the answer is a bare list; the chunk is ~100th for both retrievers.
  - **C10** (BM135): the code is in 15 chunks; tables and "refer to department" stubs fill the top 5 and the
    first description is at rank 10. A reranker would only help if it sees more than 5 candidates.
- **Out-of-scope questions** score 0.61–0.73 cosine, overlapping the lowest in-scope natural-language scores
  (0.70–0.81), so a cosine threshold alone will not separate them cleanly.

**Decision (pre-registered rule):** routed top-5 = **28/30 (≥ 27)** and top-1 = **15/30 (≤ 21)** →
**likely ranking problem; investigate reranking in Stage 6.** The failure analysis adds two points for Stage 6 to
weigh: the tie-break rule is a cheaper, known cause of ~5 losses, and two misses are retrieval problems that
reranking the top 5 cannot fix.

**Limitations**
- One grader (me), blind to method and rank. A stratified sample of 20 grades is in `eval/stage5_spot_check.txt`
  for independent checking.
- 10 questions per category: differences of one or two questions are within noise.
- E3 is ambiguous in the yearbooks; it is scored strictly against the frozen answer key.